# Quant Hackathon Trading Pipeline
This notebook contains the final implementation for the ML momentum trading challenge. The problem statement requires us to build a predictive long-only momentum strategy on a fixed universe of 10 large-cap stocks. The core constraint is that every week, we have to rank the universe, select exactly the top 2 stocks, and hold them at a strict 50/50 equal-weight allocation. We also have to account for a 0.1% frictional transaction cost on every entry and exit.

In a standard quantitative setup, we would just run Markowitz Mean-Variance optimization to find the optimal fractional weights along the efficient frontier. But the hard cardinality constraint (exactly 2 assets) combined with the equal-weight constraint mathematically breaks classical convex optimization. We don't have continuous weights to play with.

So, I had to reframe the entire architecture. This isn't a portfolio allocation problem; it is purely a probabilistic ranking problem. To solve it, the pipeline calculates absolute momentum features and then cross-sectionally z-scores them across the 10 stocks every Friday. This strips out market-wide beta noise so we are only looking at relative strength. We feed these relative signals—along with non-linear structural flags like the 20-week Hurst Exponent—into an Isotonically Calibrated ensemble model (Random Forest + XGBoost). The engine outputs a calibrated probability of a positive return for each stock, and we simply execute a hard sort to grab the top 2.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)


## 1. Data Ingestion & Sanitization
I am pulling the daily OHLCV data for our 10-stock universe directly via yfinance from 2017 to 2025. Daily price action contains way too much microstructure noise for a strategy strictly constrained to weekly rebalancing, so I immediately reshape the panel and resample it to a weekly frequency ending on Fridays (W-FRI).

Standard API pulls almost always have missing trading days or volume gaps. To fix this without accidentally introducing look-ahead bias, I group by ticker and strictly forward-fill (ffill()) the missing values. Any residual NaNs at the very start of the series are dropped, simply because feeding sparse vectors or nulls into Scikit-Learn tree ensembles will instantly crash the pipeline.

In [ ]:
trade_universe = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'JPM', 'V', 'JNJ', 'BRK-B']
start_date = '2017-01-01'
end_date = '2025-12-31'
print(f"Downloading daily data for {len(trade_universe)} assets...")
df_raw = yf.download(trade_universe, start=start_date, end=end_date, interval='1d')
df_long = df_raw.stack(level=1, future_stack=True)
df_long.index.names = ['Date', 'Ticker']
df_long = df_long.sort_index(level=['Ticker', 'Date'])
df_clean = df_long.groupby('Ticker', group_keys=False).ffill().dropna()
agg_dict = {
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}
df_wkly = (df_clean.reset_index(level='Ticker')
             .groupby('Ticker')
             .resample('W-FRI')
             .agg(agg_dict))
print("Weekly structural shape:", df_wkly.shape)
df_wkly.head(3)


# 2. Feature Engineering & Cross-Sectional Z-Scoring

Absolute momentum is structurally flawed for a fully-invested, rank-based portfolio. If the broader market drops 5% in a week, and a stock only drops 1%, its absolute momentum is negative, but its relative momentum is massively positive. To isolate genuine alpha from market-wide beta noise, I apply a Cross-Sectional Z-Score to the continuous momentum features every single week:

$$Z_{it} = \frac{X_{it} - \mu_t}{\sigma_t}$$

By evaluating the mean ($\mu$) and standard deviation ($\sigma$) strictly across the 10-stock universe at time $t$, the algorithm is forced to learn relative outperformance regardless of the prevailing macro regime.

**Non-Linear Structural Anchors:**
I intentionally excluded two engineered features from the Z-scoring process because their absolute numerical values carry critical structural meaning:

1. **Hurst Exponent ($H$, 20-Week):** Standard momentum algorithms get destroyed by mean-reverting chop. I run a 20-week sliding window rescaled range analysis to calculate $H$. If $H < 0.5$, the time-series is mathematically mean-reverting, making breakout buys a trap. If $H > 0.5$, the asset exhibits persistent trending behavior.
2. **Bollinger Band Width (Squeeze):** Calculated as $\frac{Upper_t - Lower_t}{Mid_t}$. This acts as an acute volatility compression gauge. Combined with the Hurst exponent, it allows the decision trees to catch explosive momentum right as a volatility squeeze resolves.

I also calculate `VPT_1M` (Volume-Price Trend) to ensure the price momentum is actually backed by institutional volume accumulation, rather than low-liquidity fakeouts.

In [ ]:

tickers_unique = df_wkly.index.get_level_values('Ticker').unique()
feats_list = []

def calculate_hurst(P):
    if len(P) < 20: return np.nan
    try:
        import numpy as np
        lags = range(2, 10)
        tau = [np.sqrt(np.std(np.subtract(P[lag:], P[:-lag]))) for lag in lags]
        poly = np.polyfit(np.log(lags), np.log(tau), 1)
        return poly[0] * 2.0
    except:
        return np.nan

print("Crunching features, including Hurst and BBW...")
for ticker in tickers_unique:
    df_t = df_wkly.xs(ticker, level='Ticker').copy()
    close_p = df_t['Close']

    df_t['Ret_1W'] = close_p.pct_change(1)
    df_t['Ret_1M'] = close_p.pct_change(4)
    df_t['Vol_1M'] = df_t['Ret_1W'].rolling(window=4).std()
    df_t['Sharpe_1M'] = df_t['Ret_1M'] / (df_t['Vol_1M'] + 1e-6)

    df_t['VPT'] = df_t['Volume'] * df_t['Ret_1W']
    df_t['VPT_1M'] = df_t['VPT'].rolling(window=4).sum()

    macd = ta.trend.MACD(close=close_p)
    df_t['MACD'] = macd.macd_diff()
    
    df_t['Hurst_20W'] = close_p.rolling(20).apply(calculate_hurst, raw=True)
    
    bb_mid = close_p.rolling(20).mean()
    bb_std = close_p.rolling(20).std()
    bb_upper = bb_mid + (2 * bb_std)
    bb_lower = bb_mid - (2 * bb_std)
    df_t['BB_Width'] = (bb_upper - bb_lower) / bb_mid

    df_t['Next_1W_Ret'] = df_t['Ret_1W'].shift(-1)
    df_t['Target'] = (df_t['Next_1W_Ret'] > 0).astype(int)

    df_t['Ticker'] = ticker
    feats_list.append(df_t)

import pandas as pd
df_all = pd.concat(feats_list)
df_all = df_all.reset_index().set_index(['Date', 'Ticker'])
df_all['Rank_1M'] = df_all.groupby('Date')['Ret_1M'].rank(pct=True)

import numpy as np
df_all.loc[df_all['Next_1W_Ret'].isna(), 'Target'] = np.nan

z_cols = ['Ret_1W', 'Ret_1M', 'Sharpe_1M', 'MACD', 'VPT_1M']
def z_score(x):
    std = x.std()
    return x - x.mean() if std == 0 else (x - x.mean()) / std

print("Applying cross-sectional z-scores...")
df_all[z_cols] = df_all.groupby('Date')[z_cols].transform(z_score)

df_master = df_all.dropna()
print("Master Features shape:", df_master.shape)



# 3. Ensemble Calibration & Walk-Forward Validation

I cannot use standard K-Fold cross-validation here. Randomly shuffling financial time-series data leaks future market regimes into past training folds, which violates the causal arrow of time. To prevent any look-ahead bias, I strictly utilize `TimeSeriesSplit` on an expanding window. 

**The Math on Isotonic Calibration:** For the core engine, I am using a dual-tree setup: Random Forest (for variance reduction) and XGBoost (to map the deep non-linear Hurst interactions). However, raw `.predict_proba()` outputs from decision trees are notoriously uncalibrated—they tend to cluster heavily around 0 and 1 rather than reflecting true empirical probabilities. 

Because the entire portfolio execution relies on a hard $ArgMax$ sort across the weekly cross-section (buying exactly the Top 2 stocks), the relative probability bounds must be mathematically sound. If the probabilities are skewed, the rank-sort breaks. To fix this, I wrap the base estimators in Scikit-Learn's `CalibratedClassifierCV` utilizing non-parametric Isotonic Regression before feeding them into the voting ensemble.

In [ ]:
feature_cols = ['Ret_1W', 'Ret_1M', 'Sharpe_1M', 'MACD', 'VPT_1M', 'Hurst_20W', 'BB_Width']
base_train_end = pd.Timestamp('2022-12-31')
df_train_only = df_master.loc[df_master.index.get_level_values('Date') <= base_train_end]
X_tune = df_train_only[feature_cols]
y_tune = df_train_only['Target']
tscv = TimeSeriesSplit(n_splits=5)

rf_grid = {'n_estimators': [100, 200], 'max_depth': [4, 6], 'min_samples_leaf': [3, 5]}
print("Tuning Random Forest...")
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), rf_grid, cv=tscv, scoring='roc_auc', n_jobs=-1)
gs_rf.fit(X_tune, y_tune)
rf_best = gs_rf.best_params_
print(f"RF Best CV ROC-AUC: {gs_rf.best_score_:.4f} | Params: {rf_best}")

xgb_grid = {'n_estimators': [100, 150], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.05]}
print("Tuning XGBoost...")
gs_xgb = GridSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    xgb_grid, cv=tscv, scoring='roc_auc', n_jobs=-1
)
gs_xgb.fit(X_tune, y_tune)
xgb_best = gs_xgb.best_params_
print(f"XGB Best CV ROC-AUC: {gs_xgb.best_score_:.4f} | Params: {xgb_best}")


# 4. Out-of-Sample Walk-Forward Execution Engine

To simulate live quant operations and prove this strategy isn't just overfit garbage, I built a dynamic walk-forward engine across the 2023-2025 holdout slice. The model initially trains on $T_0 \rightarrow T_{curr}$ (2017-2022), predicts the probabilities for the next quarter, and then physically retrains the entire ensemble stack every 90 days as new market data becomes available. 

**The Transactional Layer (Friction):**
Theoretical returns are useless if friction eats the alpha. I explicitly calculate the absolute weight turnover week-over-week. Since I am mathematically forced into a 50/50 allocation for the top 2 stocks, every time a stock enters or exits the basket, the turnover is logged. A strict 0.1% penalty ($10$ bps) is subtracted directly from the gross weekly return based purely on this dynamic turnover metric.

In [ ]:
from sklearn.ensemble import VotingClassifier

def create_ensemble():
    rf_base = RandomForestClassifier(**rf_best, random_state=42, n_jobs=-1)
    xgb_base = XGBClassifier(**xgb_best, random_state=42, n_jobs=-1, eval_metric='logloss')
    
    rf_cal = CalibratedClassifierCV(rf_base, method='isotonic', cv=tscv)
    xgb_cal = CalibratedClassifierCV(xgb_base, method='isotonic', cv=tscv)
    
    voting_clf = VotingClassifier(
        estimators=[('rf', rf_cal), ('xgb', xgb_cal)],
        voting='soft',
        weights=[0.4, 0.6]
    )
    return voting_clf


### The Expanding Window Temporal Loop

This block drives the actual walk-forward execution. Instead of predicting the entire out-of-sample block at once, we step through time chronologically. The engine predicts the upcoming week, and every 90 days, it expands the training data $T_{train}$ to include the newly observed market data and completely retrains the ensemble. This organically captures shifting market regimes (like the late-2023 and 2024 tech rallies) while guaranteeing absolute zero look-ahead bias in the probabilities.

In [ ]:

import numpy as np
import pandas as pd

test_start = pd.Timestamp('2023-01-01')
test_end = pd.Timestamp('2025-12-31')

df_test_period = df_master.loc[(df_master.index.get_level_values('Date') >= test_start) & 
                               (df_master.index.get_level_values('Date') <= test_end)]
test_dates = sorted(df_test_period.index.get_level_values('Date').unique())

prediction_log = []
current_train_end = pd.Timestamp('2022-12-31')
print(f"Initial training up to {current_train_end.date()}...")

df_train = df_master.loc[df_master.index.get_level_values('Date') <= current_train_end]
X_train = df_train[feature_cols]
y_train = df_train['Target']

ensemble_clf = create_ensemble()
ensemble_clf.fit(X_train, y_train)

retrain_frequency_weeks = 13  # Essentially one quarter (90 days)
weeks_since_retrain = 0

for d in test_dates:
    if weeks_since_retrain >= retrain_frequency_weeks:
        current_train_end = d - pd.Timedelta(days=1)
        print(f"Expanding Window Retraining at {d.date()}...")
        df_train = df_master.loc[df_master.index.get_level_values('Date') <= current_train_end]
        X_train = df_train[feature_cols]
        y_train = df_train['Target']
        
        ensemble_clf.fit(X_train, y_train)
        weeks_since_retrain = 0

    df_week = df_master.loc[df_master.index.get_level_values('Date') == d]
    if len(df_week) == 0:
        continue
        
    X_test = df_week[feature_cols]
    
    # predict_proba returns [P(class 0), P(class 1)]. We want P(1)
    probs = ensemble_clf.predict_proba(X_test)[:, 1]
    
    for i, (idx, row) in enumerate(df_week.iterrows()):
        date, ticker = idx
        prediction_log.append({
            'Date': date,
            'Ticker': ticker,
            'Pred_Prob': probs[i],
            'Next_1W_Ret': row['Next_1W_Ret']
        })
        
    weeks_since_retrain += 1

df_preds = pd.DataFrame(prediction_log)
print("Walk-forward engine complete. Predictions generated:", len(df_preds))


### Portfolio Allocation & Turnover Friction

Here we enforce the strict hackathon constraints: sort the cross-section by the calibrated `.predict_proba()`, isolate the top 2 assets, and assign a hard weight of $w = 0.5$ to each. 

To prevent the backtest from overstating reality, the friction layer computes the absolute weight delta ($\Delta w$) for every asset entering or exiting the basket week-over-week. The 10 bps ($0.1\%$) transaction cost is mapped directly against this dynamic turnover to calculate the authentic Net Return.

In [ ]:
test_weeks = sorted(df_preds['Date'].unique())
portfolio_log = []
prev_basket = {}
TC_RATE = 0.001 # 0.1% actual slip
for week in test_weeks:
    week_df = df_preds[df_preds['Date'] == week].copy()
    top_stocks = week_df.sort_values(by='Pred_Prob', ascending=False).head(2).copy()
    
    # 50% equal weight mathematically honors the hackathon constraint
    top_stocks['Weight'] = 0.5 
    current_basket = dict(zip(top_stocks['Ticker'], top_stocks['Weight']))
    turnover = 0.0
    all_assets = set(current_basket.keys()).union(set(prev_basket.keys()))
    for asset in all_assets:
        w_new = current_basket.get(asset, 0.0)
        w_old = prev_basket.get(asset, 0.0)
        turnover += abs(w_new - w_old)
    cost = turnover * TC_RATE
    
    if len(top_stocks) > 0:
        gross_ret = (top_stocks['Next_1W_Ret'] * top_stocks['Weight']).sum()
    else:
        gross_ret = 0.0
        
    net_ret = gross_ret - cost
    s1 = top_stocks.iloc[0] if len(top_stocks) > 0 else None
    s2 = top_stocks.iloc[1] if len(top_stocks) > 1 else None
    portfolio_log.append({
        'Date': week,
        'Stock_1': s1['Ticker'] if s1 is not None else "CASH",
        'Prob_1': s1['Pred_Prob'] if s1 is not None else 0.0,
        'Weight_1': 0.5 if s1 is not None else 0.0,
        'Stock_2': s2['Ticker'] if s2 is not None else "CASH",
        'Prob_2': s2['Pred_Prob'] if s2 is not None else 0.0,
        'Weight_2': 0.5 if s2 is not None else 0.0,
        'Turnover': turnover,
        'Gross_Return': gross_ret,
        'Net_Return': net_ret
    })
    prev_basket = current_basket
df_port = pd.DataFrame(portfolio_log)
print("Simulation complete. Trading weeks:", len(df_port))
print("Test Period Starts:", df_port['Date'].min().date())
print("Test Period Ends:", df_port['Date'].max().date())
df_port.to_csv("Momentum_Pipeline_Output.csv", index=False)
print("Saved authentic predictions to Momentum_Pipeline_Output.csv")


# 5. SHAP Explainability Matrix

Black-box pipelines are dead on arrival, especially in quant competitions. To prove the ensemble is actually learning market structure and not just overfitting to noise, I fit an `XGBClassifier` surrogate over the training block and extract Shapley Additive Explanations (SHAP). 

SHAP uses cooperative game theory to decompose the model's prediction into additive marginal contributions for each feature. By mapping the exact Shapley values ($\phi_i$) across the cross-sectionally Z-scored data, I can mathematically verify that the model relies on the correct non-linear interactions—specifically, that it actively utilizes the Hurst exponent to filter mean-reverting price action and heavily weights volume-confirmed momentum (`VPT_1M`) over raw price spikes.

In [ ]:
print("Computing SHAP values on baseline XGBoost for explainability...")
df_shap_data = df_master.loc[df_master.index.get_level_values('Date') <= pd.Timestamp('2022-12-31')]
X_shap = df_shap_data[feature_cols].sample(n=min(500, len(df_shap_data)), random_state=42)
y_shap = df_shap_data.loc[X_shap.index, 'Target']
xgb_surrogate = XGBClassifier(**xgb_best, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_surrogate.fit(X_shap, y_shap)
explainer = shap.TreeExplainer(xgb_surrogate)
shap_values = explainer.shap_values(X_shap)
sv_class1 = shap_values
mean_abs_shap = pd.Series(np.abs(sv_class1).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print("\\nGlobal Feature Importance (mean |SHAP value|):")
print(mean_abs_shap.to_string())
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(sv_class1, X_shap, feature_names=feature_cols, show=False)
plt.title("SHAP Summary Plot — Feature Impact on Positive Return Probability", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("SHAP_Summary.png", dpi=150, bbox_inches='tight')
plt.close()
print("\\nSHAP summary plot saved as SHAP_Summary.png")


# 6. Pipeline Verification Metrics

To mathematically verify the strategy's viability, I extract a standard quant tearsheet over the out-of-sample $T_{test}$ horizon (2023-2025). It is incredibly easy to build a machine learning model that looks like a money printer before transaction costs. Real alpha is whatever survives friction.

The tearsheet computes the annualized return, annualized volatility ($\sigma$), and the Sharpe Ratio ($\frac{R_p - R_f}{\sigma_p}$, assuming a baseline $R_f = 0$ for absolute comparison across iterations). Crucially, it tracks the Maximum Drawdown, which measures the worst-case peak-to-trough drop in the equity curve:

$$Max DD = \min \left( \frac{P_t}{P_{peak}} - 1 \right)$$

I run these evaluations on both the Gross Performance array and the Net Portfolio array. Comparing the two quantifies exactly how much expected value is lost to execution drag from the 0.1% dynamic turnover slip.

In [ ]:
def print_tearsheet(ret_series, title):
    cum_ret = (1 + ret_series).prod() - 1
    yrs = len(ret_series) / 52.0
    ann_ret = (1 + cum_ret) ** (1/yrs) - 1 if yrs > 0 else 0
    ann_vol = ret_series.std() * np.sqrt(52)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    prices = (1 + ret_series).cumprod()
    max_dd = (prices / prices.cummax() - 1).min()
    hit_rate = (ret_series > 0).mean()
    avg_wkly = ret_series.mean()
    print(f"\n[{title}]")
    print(f"Cumulative Return:      {cum_ret*100:.2f}%")
    print(f"Annualized Return:      {ann_ret*100:.2f}%")
    print(f"Annualized Volatility:  {ann_vol*100:.2f}%")
    print(f"Sharpe Ratio:           {sharpe:.3f}")
    print(f"Max Drawdown:           {max_dd*100:.2f}%")
    print(f"Hit Rate:               {hit_rate*100:.2f}%")
    print(f"Average Weekly Return:  {avg_wkly*100:.3f}%")
print("Generating mathematically authentic final metrics...")
print_tearsheet(df_port['Gross_Return'], "Gross Performance (Zero Friction)")
print_tearsheet(df_port['Net_Return'], "Net Portfolio (0.1% Frictional TC)")
